# 13 — Data source audit and API probe

This notebook does not train a model. It audits which data sources are available, which endpoints work, how many rows they return, and which sources are worth using next.

Source classes checked:

1. **Open historical results**
   - international results;
   - goalscorers;
   - shootouts;
   - StatsBomb open data.

2. **Odds sources**
   - The Odds API current odds;
   - The Odds API historical odds;
   - current sport keys;
   - candidate historical sport keys.

3. **Match / fixtures / standings APIs**
   - football-data.org;
   - API-Football / API-Sports;
   - Sportmonks, if a token is available.

4. **Recommendation layer**
   - which sources to use in the production-style chain;
   - which sources are worth paying for;
   - where remaining API credits should be spent.

Output:
- `data/processed/13_data_source_audit/source_inventory.csv`
- `data/processed/13_data_source_audit/probe_results.csv`
- `data/processed/13_data_source_audit/recommended_source_plan.json`
- `data/processed/13_data_source_audit_report_bundle.zip`


In [ ]:
# Cell 1. Paste API keys here if available.

# The Odds API:
ODDS_API_KEY = "PASTE_THE_ODDS_API_KEY_HERE"

# API-Football / API-Sports:
API_FOOTBALL_KEY = "PASTE_API_FOOTBALL_KEY_HERE"

# football-data.org:
FOOTBALL_DATA_ORG_TOKEN = "PASTE_FOOTBALL_DATA_ORG_TOKEN_HERE"

# Sportmonks:
SPORTMONKS_TOKEN = "PASTE_SPORTMONKS_TOKEN_HERE"


In [ ]:
# Cell 2. Imports and helpers.

from pathlib import Path
import json
import time
import re
import zipfile

import requests
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = PROCESSED_DIR / "13_data_source_audit"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()

def has_key(value):
    if value is None:
        return False
    value = str(value).strip()
    if value == "":
        return False
    if value.startswith("PASTE_"):
        return False
    return True

def mask_key(value):
    if not has_key(value):
        return "missing"
    value = str(value)
    return f"present_len_{len(value)}_last4_{value[-4:]}"

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    return path

def safe_get_json(url, params=None, headers=None, timeout=60):
    try:
        response = requests.get(
            url,
            params=params or {},
            headers=headers or {},
            timeout=timeout,
        )
        try:
            data = response.json()
        except Exception:
            data = {"raw_text": response.text[:3000]}
        return response.status_code, data, dict(response.headers)
    except Exception as exc:
        return -1, {"error": str(exc)}, {}

def safe_read_csv(url_or_path, nrows=None):
    try:
        df = pd.read_csv(url_or_path, nrows=nrows)
        return True, df, ""
    except Exception as exc:
        return False, pd.DataFrame(), str(exc)

def row_count_from_payload(data):
    if isinstance(data, list):
        return len(data)
    if isinstance(data, dict):
        for key in [
            "data",
            "response",
            "matches",
            "competitions",
            "fixtures",
            "result",
        ]:
            value = data.get(key)
            if isinstance(value, list):
                return len(value)
    return 0

def error_from_payload(data):
    if isinstance(data, dict):
        for key in ["message", "error", "errors"]:
            if key in data:
                return str(data.get(key))[:500]
    return ""

def write_payload(name, payload):
    safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    path = OUT_DIR / "raw_payloads" / f"{safe_name}.json"
    save_json(path, payload)
    return path

print("Setup OK.")


In [ ]:
# Cell 3. Source inventory.

sources = [
    {
        "source": "martj42 international_results",
        "type": "open_csv",
        "cost": "free",
        "primary_use": "historical international match results, goalscorers, shootouts",
        "model_use": "Elo, form, score model, outcome training",
        "priority": "core",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    },
    {
        "source": "martj42 goalscorers",
        "type": "open_csv",
        "cost": "free",
        "primary_use": "goal scorers, penalty flags, own goals",
        "model_use": "player contribution proxy, scorer availability historical patterns",
        "priority": "medium",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv",
    },
    {
        "source": "martj42 shootouts",
        "type": "open_csv",
        "cost": "free",
        "primary_use": "penalty shootout results",
        "model_use": "knockout simulation, not 90-minute 1X2",
        "priority": "medium",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv",
    },
    {
        "source": "StatsBomb open data competitions",
        "type": "open_json",
        "cost": "free with attribution requirements",
        "primary_use": "event data availability check",
        "model_use": "xG / event features where available, especially World Cup 2022",
        "priority": "medium",
        "url": "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json",
    },
    {
        "source": "The Odds API current odds",
        "type": "paid_api",
        "cost": "paid credits",
        "primary_use": "current odds, best prices, market probabilities",
        "model_use": "current predictions and paper picks",
        "priority": "core",
        "url": "https://api.the-odds-api.com/v4/sports/{sport}/odds",
    },
    {
        "source": "The Odds API historical odds",
        "type": "paid_api",
        "cost": "paid credits, historical endpoint",
        "primary_use": "historical market probabilities and CLV backtest",
        "model_use": "edge validation, market anchored correction",
        "priority": "core",
        "url": "https://api.the-odds-api.com/v4/historical/sports/{sport}/odds",
    },
    {
        "source": "football-data.org",
        "type": "api",
        "cost": "free plus paid tiers",
        "primary_use": "fixtures, results, competitions, some odds/stats by tier",
        "model_use": "schedule validation, competitions, results cross-check",
        "priority": "medium",
        "url": "https://api.football-data.org/v4/competitions",
    },
    {
        "source": "API-Football / API-Sports",
        "type": "paid_api",
        "cost": "free limited, paid plans",
        "primary_use": "fixtures, lineups, injuries, standings, odds, events, stats",
        "model_use": "pre-match enrichments, lineups, injuries, standings",
        "priority": "high_if_paid",
        "url": "https://v3.football.api-sports.io",
    },
    {
        "source": "Sportmonks Football API",
        "type": "paid_api",
        "cost": "paid plans",
        "primary_use": "fixtures, lineups, injuries, odds, predictions, stats",
        "model_use": "alternative/complement to API-Football",
        "priority": "high_if_paid",
        "url": "https://api.sportmonks.com/v3/football",
    },
]

source_inventory = pd.DataFrame(sources)

source_inventory.to_csv(
    OUT_DIR / "source_inventory.csv",
    index=False,
)

display(source_inventory)


In [ ]:
# Cell 4. Probe open CSV / JSON sources.

probe_rows = []

open_csv_sources = [
    {
        "name": "international_results_results",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    },
    {
        "name": "international_results_goalscorers",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv",
    },
    {
        "name": "international_results_shootouts",
        "url": "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv",
    },
]

for item in open_csv_sources:
    ok, df, error = safe_read_csv(item["url"], nrows=None)
    out_path = None

    if ok:
        out_path = RAW_DIR / f"{item['name']}.csv"
        df.to_csv(out_path, index=False)

    probe_rows.append({
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "source": item["name"],
        "probe_type": "open_csv",
        "status": "ok" if ok else "failed",
        "rows": int(len(df)) if ok else 0,
        "columns": ",".join(df.columns) if ok else "",
        "error": error,
        "saved_path": str(out_path) if out_path else "",
    })

# StatsBomb competitions JSON.
status, data, headers = safe_get_json(
    "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json"
)

payload_path = write_payload(
    "statsbomb_competitions",
    {
        "status": status,
        "data_sample": data[:10] if isinstance(data, list) else data,
    },
)

probe_rows.append({
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "source": "statsbomb_competitions",
    "probe_type": "open_json",
    "status": "ok" if status == 200 else "failed",
    "rows": row_count_from_payload(data),
    "columns": "",
    "error": error_from_payload(data),
    "saved_path": str(payload_path),
})

open_probe = pd.DataFrame(probe_rows)

open_probe.to_csv(
    OUT_DIR / "open_source_probe_results.csv",
    index=False,
)

display(open_probe)


In [ ]:
# Cell 5. Probe The Odds API.

odds_probe_rows = []

if has_key(ODDS_API_KEY):
    # Sports list.
    status, data, headers = safe_get_json(
        "https://api.the-odds-api.com/v4/sports",
        params={"apiKey": ODDS_API_KEY},
    )

    payload_path = write_payload(
        "the_odds_api_sports",
        {
            "status": status,
            "headers": headers,
            "data": data,
        },
    )

    soccer_sports = []

    if isinstance(data, list):
        for item in data:
            if str(item.get("key", "")).startswith("soccer"):
                soccer_sports.append(item)

    soccer_sports_df = pd.DataFrame(soccer_sports)
    soccer_sports_df.to_csv(
        OUT_DIR / "the_odds_api_soccer_sports.csv",
        index=False,
    )

    odds_probe_rows.append({
        "source": "the_odds_api",
        "probe": "sports",
        "status_code": status,
        "rows": row_count_from_payload(data),
        "x_requests_remaining": headers.get("x-requests-remaining"),
        "x_requests_used": headers.get("x-requests-used"),
        "x_requests_last": headers.get("x-requests-last"),
        "error": error_from_payload(data),
        "saved_path": str(payload_path),
    })

    # Candidate current/historical sport keys.
    candidate_sport_keys = [
        "soccer_fifa_world_cup",
        "soccer_uefa_euro",
        "soccer_uefa_european_championship",
        "soccer_copa_america",
        "soccer_africa_cup_of_nations",
        "soccer_afc_asian_cup",
        "soccer_concacaf_gold_cup",
        "soccer_uefa_nations_league",
        "soccer_international_friendlies",
    ]

    for sport_key in candidate_sport_keys:
        # Current odds smoke test.
        status, data, headers = safe_get_json(
            f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds",
            params={
                "apiKey": ODDS_API_KEY,
                "regions": "eu",
                "markets": "h2h",
                "oddsFormat": "decimal",
                "dateFormat": "iso",
            },
        )

        odds_probe_rows.append({
            "source": "the_odds_api",
            "probe": f"current_odds_{sport_key}",
            "status_code": status,
            "rows": row_count_from_payload(data),
            "x_requests_remaining": headers.get("x-requests-remaining"),
            "x_requests_used": headers.get("x-requests-used"),
            "x_requests_last": headers.get("x-requests-last"),
            "error": error_from_payload(data),
            "saved_path": "",
        })

        time.sleep(0.12)

else:
    odds_probe_rows.append({
        "source": "the_odds_api",
        "probe": "skipped",
        "status_code": None,
        "rows": 0,
        "x_requests_remaining": None,
        "x_requests_used": None,
        "x_requests_last": None,
        "error": "ODDS_API_KEY missing",
        "saved_path": "",
    })

the_odds_probe = pd.DataFrame(odds_probe_rows)

the_odds_probe.to_csv(
    OUT_DIR / "the_odds_api_probe_results.csv",
    index=False,
)

display(the_odds_probe)


In [ ]:
# Cell 6. Probe football-data.org.

football_data_rows = []

if has_key(FOOTBALL_DATA_ORG_TOKEN):
    headers = {"X-Auth-Token": FOOTBALL_DATA_ORG_TOKEN}

    endpoints = [
        ("competitions", "https://api.football-data.org/v4/competitions"),
        ("worldcup_matches", "https://api.football-data.org/v4/competitions/WC/matches"),
        ("euro_matches", "https://api.football-data.org/v4/competitions/EC/matches"),
    ]

    for name, url in endpoints:
        status, data, response_headers = safe_get_json(
            url,
            headers=headers,
        )

        payload_path = write_payload(
            f"football_data_org_{name}",
            {
                "status": status,
                "headers": response_headers,
                "data": data,
            },
        )

        football_data_rows.append({
            "source": "football-data.org",
            "probe": name,
            "status_code": status,
            "rows": row_count_from_payload(data),
            "error": error_from_payload(data),
            "saved_path": str(payload_path),
        })

        time.sleep(0.2)
else:
    football_data_rows.append({
        "source": "football-data.org",
        "probe": "skipped",
        "status_code": None,
        "rows": 0,
        "error": "FOOTBALL_DATA_ORG_TOKEN missing",
        "saved_path": "",
    })

football_data_probe = pd.DataFrame(football_data_rows)

football_data_probe.to_csv(
    OUT_DIR / "football_data_org_probe_results.csv",
    index=False,
)

display(football_data_probe)


In [ ]:
# Cell 7. Probe API-Football / API-Sports.

api_football_rows = []

if has_key(API_FOOTBALL_KEY):
    headers = {"x-apisports-key": API_FOOTBALL_KEY}
    base = "https://v3.football.api-sports.io"

    endpoints = [
        ("status", f"{base}/status", {}),
        ("countries", f"{base}/countries", {}),
        ("leagues_search_world", f"{base}/leagues", {"search": "World Cup"}),
        ("leagues_search_euro", f"{base}/leagues", {"search": "Euro"}),
        ("fixtures_worldcup_2026", f"{base}/fixtures", {"league": 1, "season": 2026}),
    ]

    for name, url, params in endpoints:
        status, data, response_headers = safe_get_json(
            url,
            params=params,
            headers=headers,
        )

        payload_path = write_payload(
            f"api_football_{name}",
            {
                "status": status,
                "headers": response_headers,
                "params": params,
                "data": data,
            },
        )

        api_football_rows.append({
            "source": "api-football",
            "probe": name,
            "status_code": status,
            "rows": row_count_from_payload(data),
            "error": error_from_payload(data),
            "saved_path": str(payload_path),
        })

        time.sleep(0.25)
else:
    api_football_rows.append({
        "source": "api-football",
        "probe": "skipped",
        "status_code": None,
        "rows": 0,
        "error": "API_FOOTBALL_KEY missing",
        "saved_path": "",
    })

api_football_probe = pd.DataFrame(api_football_rows)

api_football_probe.to_csv(
    OUT_DIR / "api_football_probe_results.csv",
    index=False,
)

display(api_football_probe)


In [ ]:
# Cell 8. Probe Sportmonks.

sportmonks_rows = []

if has_key(SPORTMONKS_TOKEN):
    base = "https://api.sportmonks.com/v3/football"

    endpoints = [
        ("leagues", f"{base}/leagues", {}),
        ("fixtures_upcoming", f"{base}/fixtures/upcoming", {}),
    ]

    for name, url, params in endpoints:
        params = dict(params)
        params["api_token"] = SPORTMONKS_TOKEN

        status, data, response_headers = safe_get_json(
            url,
            params=params,
        )

        payload_path = write_payload(
            f"sportmonks_{name}",
            {
                "status": status,
                "headers": response_headers,
                "params_masked": {
                    **{k: v for k, v in params.items() if k != "api_token"},
                    "api_token": mask_key(SPORTMONKS_TOKEN),
                },
                "data": data,
            },
        )

        sportmonks_rows.append({
            "source": "sportmonks",
            "probe": name,
            "status_code": status,
            "rows": row_count_from_payload(data),
            "error": error_from_payload(data),
            "saved_path": str(payload_path),
        })

        time.sleep(0.25)
else:
    sportmonks_rows.append({
        "source": "sportmonks",
        "probe": "skipped",
        "status_code": None,
        "rows": 0,
        "error": "SPORTMONKS_TOKEN missing",
        "saved_path": "",
    })

sportmonks_probe = pd.DataFrame(sportmonks_rows)

sportmonks_probe.to_csv(
    OUT_DIR / "sportmonks_probe_results.csv",
    index=False,
)

display(sportmonks_probe)


In [ ]:
# Cell 9. Build unified probe report.

all_probe_results = pd.concat(
    [
        open_probe,
        the_odds_probe,
        football_data_probe,
        api_football_probe,
        sportmonks_probe,
    ],
    ignore_index=True,
    sort=False,
)

all_probe_results.to_csv(
    OUT_DIR / "probe_results.csv",
    index=False,
)

display(all_probe_results)


In [ ]:
# Cell 10. Recommended source plan.

recommended_plan = {
    "core_chain": [
        {
            "source": "martj42 international_results",
            "use": "base historical outcomes and score training",
            "action": "keep in notebook 01",
            "priority": 1,
        },
        {
            "source": "The Odds API historical h2h",
            "use": "market probabilities, CLV, edge validation",
            "action": (
                "spend remaining credits mostly here: expand to 300-500 "
                "joined market matches and add T-7d/T-48h/T-12h/T-6h"
            ),
            "priority": 1,
        },
        {
            "source": "The Odds API current h2h",
            "use": "current WC2026 paper picks and live CLV ledger",
            "action": "run notebook 04 daily and near kickoff",
            "priority": 1,
        },
    ],
    "paid_enrichment_candidates": [
        {
            "source": "API-Football",
            "use": "fixtures, lineups, injuries, standings, events, stats",
            "decision_rule": (
                "buy if probes confirm World Cup 2026 fixtures, lineups, "
                "injuries, and team stats are available on your plan"
            ),
            "priority": 2,
        },
        {
            "source": "Sportmonks",
            "use": "alternative provider for lineups, injuries, odds, predictions",
            "decision_rule": (
                "consider only if API-Football coverage is weak or if "
                "Sportmonks plan gives better international coverage"
            ),
            "priority": 3,
        },
    ],
    "free_enrichment_candidates": [
        {
            "source": "StatsBomb open data",
            "use": "event/xG features for competitions available in open data",
            "caveat": "not comprehensive for all current matches; good for feature research",
            "priority": 3,
        },
        {
            "source": "football-data.org",
            "use": "schedule/results cross-check and competitions",
            "caveat": "useful but may not beat The Odds API + martj42 for this chain",
            "priority": 3,
        },
    ],
    "do_not_prioritize_yet": [
        "totals/spreads markets before proving 1X2 draw edge",
        "player props",
        "expensive multi-region historical odds before we validate h2h edge",
    ],
    "next_notebook_recommendation": {
        "name": "14_budgeted_historical_expansion_v2",
        "goal": (
            "Use probe_results to expand historical h2h odds only for "
            "sport keys that worked, add more snapshots, and keep a "
            "hard credit budget."
        ),
        "budget_suggestion_credits": 6000,
    },
}

save_json(
    OUT_DIR / "recommended_source_plan.json",
    recommended_plan,
)

print(json.dumps(recommended_plan, indent=2, ensure_ascii=False))


In [ ]:
# Cell 11. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "source_inventory_rows": int(len(source_inventory)),
    "probe_rows": int(len(all_probe_results)),
    "keys": {
        "odds_api": mask_key(ODDS_API_KEY),
        "api_football": mask_key(API_FOOTBALL_KEY),
        "football_data_org": mask_key(FOOTBALL_DATA_ORG_TOKEN),
        "sportmonks": mask_key(SPORTMONKS_TOKEN),
    },
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "13_data_source_audit_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
